# 02 — Data Preprocessing
Clean data, validate numeric fields, handle missing values and create model features.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
ROOT = Path.cwd().parent
RAW = ROOT/"data"/"raw"/"water_quality_raw.csv"
PROC = ROOT/"data"/"processed"
MODELS = ROOT/"models"

source=RAW if RAW.exists() else PROC/"exploration_snapshot.csv"
if not source.exists(): raise FileNotFoundError("Run notebook 01 or add data/raw/water_quality_raw.csv")
df=pd.read_csv(source); df.columns=[c.strip().lower().replace(" ","_") for c in df.columns]
candidates=["ph","turbidity","tds","temperature","dissolved_oxygen","conductivity"]
features=[c for c in candidates if c in df.columns]
if len(features)<3: raise ValueError(f"Need at least 3 supported features; found {features}")
for c in features: df[c]=pd.to_numeric(df[c],errors="coerce")
df=df.drop_duplicates(); df[features]=df[features].replace([np.inf,-np.inf],np.nan)
df[features]=df[features].fillna(df[features].median()); clean=df.dropna(subset=features).copy()
PROC.mkdir(parents=True,exist_ok=True); clean.to_csv(PROC/"water_quality_clean.csv",index=False)
print("Features:",features,"Shape:",clean.shape)


In [ ]:
from sklearn.preprocessing import StandardScaler
X=clean[features]; scaler=StandardScaler(); Xs=scaler.fit_transform(X)
pd.DataFrame(Xs,columns=features).to_csv(PROC/"water_quality_features.csv",index=False)
display(pd.DataFrame(Xs,columns=features).head())
